# 🏕️ CampingTN — Budget Prediction ML Model
## Tunisia Camping Budget & Equipment Planner

This notebook builds a **Random Forest + Gradient Boosting ensemble** model to predict camping budgets in Tunisia based on:
- Governorate (24 Tunisian regions)
- Site type (Forest, Coastal, Desert)
- Number of persons & days
- Season and weather conditions
- Accommodation type

**Dataset**: Tunisia official camping centers survey 2002–2017 + synthetic cost data

**Output**: Trained model exported as `camping_budget_model.pkl` for Flask API integration

In [ ]:
# Install dependencies
!pip install pandas numpy scikit-learn matplotlib seaborn joblib xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

plt.style.use('seaborn-v0_8-whitegrid')
print('✅ Libraries loaded')

## 1. Load and Explore Official Dataset

In [ ]:
# Load official camping centers data
camping_raw = {
    'nb': [1, 3, 4, 7, 11, 12, 15, 16, 17, 18, 20, 21, 22],
    'name': [
        'Centre El Hbibia', 'Centre Chat Mami', 'Centre Errimal', 'Centre Beni Mtir',
        'Centre Ain Bousaadia', 'Centre El Salloume', 'Centre El Douirat (Mahdia)',
        'Centre Erramela', 'Centre Ain Selsla', 'Centre El Cheaanbi',
        'Centre Marsa El Kssiba', 'Centre El Douirat (Tataouine)', 'Centre Douz'
    ],
    'governorate': [
        'Manouba', 'Bizerte', 'Bizerte', 'Jendouba', 'Manouba', 'Sousse',
        'Mahdia', 'Sfax', 'Kasserine', 'Kasserine', 'Médenine', 'Tataouine', 'Kébili'
    ],
    'region': [
        'District-Tunis', 'Nord-Est', 'Nord-Est', 'Nord-Ouest', 'Nord-Ouest',
        'Centre-Est', 'Centre-Est', 'Centre-Est', 'Centre-Ouest', 'Centre-Ouest',
        'Sud-Est', 'Sud-Est', 'Sud-Ouest'
    ],
    'year_created': [1989, None, None, None, 1989, 1999, 2011, 1986, 1963, 2008, 1987, 1996, 1990],
    'capacity_buildings': [0, 30, 95, 100, 32, 30, 80, 80, 60, 36, 64, 0, 30],
    'capacity_tents': [80, 45, 0, 0, 100, 100, 0, 0, 70, 50, 200, 0, 200],
    'capacity_total': [80, 75, 95, 100, 132, 130, 80, 80, 130, 86, 264, 0, 230],
    'site_nature': [
        'FOREST', 'COASTAL', 'COASTAL', 'FOREST', 'FOREST', 'COASTAL',
        'COASTAL', 'COASTAL', 'FOREST', 'DESERT', 'COASTAL', 'DESERT', 'DESERT'
    ],
}

df_centers = pd.DataFrame(camping_raw)
print(f'Camping Centers Dataset: {df_centers.shape}')
df_centers.head()

In [ ]:
# Load evolution data 2002-2017
evolution_data = {
    'region': ['District-Tunis', 'Nord-Est', 'Nord-Ouest', 'Centre-Est', 'Centre-Ouest', 'Sud-Est', 'Sud-Ouest'],
    '2002': [2, 4, 7, 4, 1, 4, 1],
    '2004': [2, 4, 7, 4, 1, 4, 1],
    '2006': [2, 4, 7, 4, 1, 4, 1],
    '2008': [2, 4, 7, 4, 2, 4, 1],
    '2010': [2, 4, 7, 6, 2, 4, 1],
    '2012': [2, 3, 7, 7, 2, 4, 1],
    '2014': [2, 3, 7, 7, 2, 4, 1],
    '2016': [1, 3, 7, 5, 2, 3, 1],
    '2017': [0, 0, 0, 1, 0, 0, 1],  # active/new
}
df_evo = pd.DataFrame(evolution_data)
print('Evolution 2002-2017:')
print(df_evo.to_string())

## 2. Generate Training Dataset

In [ ]:
np.random.seed(42)

# Base cost parameters (TND per person per day)
ACCOMMODATION_COSTS = {
    ('TENT', 'COASTAL'): (18, 4),
    ('TENT', 'FOREST'): (15, 3),
    ('TENT', 'DESERT'): (12, 3),
    ('BUILDING', 'COASTAL'): (35, 7),
    ('BUILDING', 'FOREST'): (28, 5),
    ('BUILDING', 'DESERT'): (32, 6),
}

GOVERNORATE_INDEX = {
    'Tunis': 1.30, 'Ariana': 1.20, 'Ben Arous': 1.20, 'Manouba': 1.00,
    'Bizerte': 1.10, 'Nabeul': 1.15, 'Zaghouan': 0.95, 'Beja': 0.90,
    'Jendouba': 0.88, 'Le Kef': 0.87, 'Siliana': 0.85,
    'Sousse': 1.20, 'Monastir': 1.20, 'Mahdia': 1.10,
    'Sfax': 1.05, 'Kairouan': 0.90, 'Kasserine': 0.85, 'Sidi Bouzid': 0.83,
    'Gabès': 0.95, 'Médenine': 1.00, 'Tataouine': 0.95,
    'Gafsa': 0.88, 'Tozeur': 1.00, 'Kébili': 0.92,
}

SEASON_MULT = {
    ('COASTAL', 'SUMMER'): 1.50, ('COASTAL', 'SPRING'): 1.10,
    ('COASTAL', 'AUTUMN'): 1.10, ('COASTAL', 'WINTER'): 0.85,
    ('DESERT', 'WINTER'): 1.30, ('DESERT', 'SPRING'): 1.30,
    ('DESERT', 'AUTUMN'): 1.30, ('DESERT', 'SUMMER'): 0.70,
    ('FOREST', 'SPRING'): 1.20, ('FOREST', 'AUTUMN'): 1.20,
    ('FOREST', 'SUMMER'): 1.10, ('FOREST', 'WINTER'): 0.90,
}

DISTANCE_KM = {
    'Tunis': 0, 'Ariana': 15, 'Ben Arous': 25, 'Manouba': 20,
    'Bizerte': 65, 'Nabeul': 80, 'Zaghouan': 60, 'Beja': 110,
    'Jendouba': 160, 'Le Kef': 175, 'Siliana': 140,
    'Sousse': 140, 'Monastir': 160, 'Mahdia': 200,
    'Sfax': 270, 'Kairouan': 155, 'Kasserine': 250, 'Sidi Bouzid': 250,
    'Gabès': 360, 'Médenine': 430, 'Tataouine': 500,
    'Gafsa': 360, 'Tozeur': 450, 'Kébili': 430,
}

TYPICAL_WEATHER = {
    'COASTAL': {'SUMMER': (32, 70), 'WINTER': (13, 80), 'SPRING': (22, 70), 'AUTUMN': (20, 72)},
    'FOREST': {'SUMMER': (30, 65), 'WINTER': (10, 85), 'SPRING': (20, 72), 'AUTUMN': (18, 75)},
    'DESERT': {'SUMMER': (44, 18), 'WINTER': (15, 42), 'SPRING': (28, 28), 'AUTUMN': (25, 25)},
}

def generate_sample(n=5000):
    govs = list(GOVERNORATE_INDEX.keys())
    site_types = ['COASTAL', 'FOREST', 'DESERT']
    seasons = ['SPRING', 'SUMMER', 'AUTUMN', 'WINTER']
    accom_types = ['TENT', 'BUILDING']
    
    rows = []
    for _ in range(n):
        gov = np.random.choice(govs)
        site = np.random.choice(site_types)
        season = np.random.choice(seasons)
        accom = np.random.choice(accom_types)
        persons = np.random.randint(1, 15)
        days = np.random.randint(1, 14)
        
        gov_idx = GOVERNORATE_INDEX[gov]
        s_mult = SEASON_MULT.get((site, season), 1.0)
        base_accom_mu, base_accom_std = ACCOMMODATION_COSTS.get((accom, site), (20, 4))
        accom_per_person_day = max(5, np.random.normal(base_accom_mu, base_accom_std))
        accom_total = accom_per_person_day * persons * days * gov_idx * s_mult
        
        food_ppd = np.random.normal(35 * gov_idx, 5)
        food_total = food_ppd * persons * days
        
        dist = DISTANCE_KM[gov]
        transport = dist * 2 * np.random.normal(0.35, 0.05)
        transport += 50 if persons > 4 else 0
        
        equip_base = {'DESERT': 45, 'FOREST': 30, 'COASTAL': 25}[site]
        equip = equip_base * persons + (days * 5 if days > 3 else 0) + np.random.normal(0, 15)
        
        misc_base = {'DESERT': 80, 'COASTAL': 60, 'FOREST': 30}[site]
        misc = (misc_base + days * 10) * min(persons, 4) + np.random.normal(0, 20)
        
        total = max(100, accom_total + food_total + transport + equip + misc)
        
        temp, humidity = TYPICAL_WEATHER[site][season]
        temp += np.random.normal(0, 2)
        humidity += np.random.normal(0, 5)
        
        rows.append({
            'governorate': gov,
            'site_type': site,
            'season': season,
            'accommodation_type': accom,
            'num_persons': persons,
            'num_days': days,
            'gov_cost_index': gov_idx,
            'season_multiplier': s_mult,
            'distance_km': dist,
            'temperature': round(temp, 1),
            'humidity': round(humidity, 1),
            'accommodation_cost': round(accom_total, 2),
            'food_cost': round(food_total, 2),
            'transport_cost': round(transport, 2),
            'equipment_cost': round(equip, 2),
            'misc_cost': round(misc, 2),
            'total_budget': round(total, 2),
        })
    return pd.DataFrame(rows)

df = generate_sample(5000)
print(f'Generated {len(df)} training samples')
df.describe()

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('CampingTN — Budget Analysis', fontsize=16, fontweight='bold')

# Budget distribution by site type
for stype, color in zip(['COASTAL', 'FOREST', 'DESERT'], ['#0077b6', '#2d6a4f', '#e9c46a']):
    axes[0,0].hist(df[df.site_type==stype]['total_budget'], alpha=0.7, label=stype, color=color, bins=40)
axes[0,0].set_title('Budget Distribution by Site Type')
axes[0,0].set_xlabel('Total Budget (TND)')
axes[0,0].legend()

# Budget by season
season_means = df.groupby('season')['total_budget'].mean().sort_values(ascending=False)
axes[0,1].bar(season_means.index, season_means.values, color=['#f59e0b', '#ef4444', '#10b981', '#0077b6'])
axes[0,1].set_title('Average Budget by Season')
axes[0,1].set_ylabel('Avg Budget (TND)')

# Budget vs persons
axes[0,2].scatter(df['num_persons'], df['total_budget'], alpha=0.1, color='#2d6a4f', s=10)
axes[0,2].set_title('Budget vs Number of Persons')
axes[0,2].set_xlabel('Persons')

# Budget vs days
axes[1,0].scatter(df['num_days'], df['total_budget'], alpha=0.1, color='#0077b6', s=10)
axes[1,0].set_title('Budget vs Number of Days')
axes[1,0].set_xlabel('Days')

# Top 10 governorates by avg budget
gov_avg = df.groupby('governorate')['total_budget'].mean().sort_values(ascending=True).tail(10)
axes[1,1].barh(gov_avg.index, gov_avg.values, color='#52b788')
axes[1,1].set_title('Top Governorates by Avg Budget')

# Cost breakdown pie
cost_cols = ['accommodation_cost', 'food_cost', 'transport_cost', 'equipment_cost', 'misc_cost']
means = df[cost_cols].mean()
axes[1,2].pie(means, labels=['Accommodation', 'Food', 'Transport', 'Equipment', 'Activities'],
              autopct='%1.1f%%', colors=['#2d6a4f', '#52b788', '#0077b6', '#e9c46a', '#f4a261'])
axes[1,2].set_title('Average Cost Breakdown')

plt.tight_layout()
plt.savefig('budget_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Analysis charts saved')

## 4. Feature Engineering & Encoding

In [ ]:
df_ml = df.copy()

# Encode categoricals
encoders = {}
for col in ['governorate', 'site_type', 'season', 'accommodation_type']:
    le = LabelEncoder()
    df_ml[col + '_enc'] = le.fit_transform(df_ml[col])
    encoders[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Feature engineering
df_ml['persons_days'] = df_ml['num_persons'] * df_ml['num_days']
df_ml['cost_per_day'] = df_ml['total_budget'] / df_ml['num_days']
df_ml['cost_per_person_day'] = df_ml['total_budget'] / (df_ml['num_persons'] * df_ml['num_days'])
df_ml['is_southern'] = df_ml['governorate'].isin(['Kébili', 'Tataouine', 'Tozeur', 'Gafsa', 'Médenine']).astype(int)
df_ml['is_coastal_region'] = df_ml['governorate'].isin(['Nabeul', 'Bizerte', 'Sousse', 'Monastir', 'Mahdia', 'Sfax']).astype(int)
df_ml['heat_index'] = df_ml['temperature'] * (1 - df_ml['humidity'] / 100)

FEATURES = [
    'governorate_enc', 'site_type_enc', 'season_enc', 'accommodation_type_enc',
    'num_persons', 'num_days', 'persons_days',
    'gov_cost_index', 'season_multiplier', 'distance_km',
    'temperature', 'humidity', 'heat_index',
    'is_southern', 'is_coastal_region'
]

TARGET = 'total_budget'

X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 5. Model Training — Ensemble Approach

In [ ]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=5,
                            min_samples_leaf=2, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)
print(f'Random Forest — MAE: {rf_mae:.2f} TND, R²: {rf_r2:.4f}')

# Train Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.08, max_depth=6,
                                subsample=0.8, random_state=42)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_r2 = r2_score(y_test, gb_pred)
print(f'Gradient Boosting — MAE: {gb_mae:.2f} TND, R²: {gb_r2:.4f}')

# Ensemble
ensemble = VotingRegressor([('rf', rf), ('gb', gb)])
ensemble.fit(X_train, y_train)
ens_pred = ensemble.predict(X_test)
ens_mae = mean_absolute_error(y_test, ens_pred)
ens_r2 = r2_score(y_test, ens_pred)
print(f'Ensemble — MAE: {ens_mae:.2f} TND, R²: {ens_r2:.4f}')

best_model = ensemble if ens_mae < rf_mae else rf
print(f'\n✅ Best model selected')

In [ ]:
# Feature importance
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)

plt.figure(figsize=(12, 6))
colors = ['#2d6a4f' if i < 5 else '#52b788' if i < 10 else '#d8f3dc' for i in range(len(importances))]
plt.bar(importances.index, importances.values, color=colors)
plt.xticks(rotation=45, ha='right')
plt.title('Feature Importance — Budget Prediction Model', fontsize=14, fontweight='bold')
plt.ylabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print('\nTop 5 most important features:')
print(importances.head())

In [ ]:
# Prediction quality
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(y_test, ens_pred, alpha=0.3, color='#2d6a4f', s=15)
ax1.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
ax1.set_xlabel('Actual Budget (TND)')
ax1.set_ylabel('Predicted Budget (TND)')
ax1.set_title(f'Actual vs Predicted\nR² = {ens_r2:.4f}, MAE = {ens_mae:.1f} TND')

residuals = y_test - ens_pred
ax2.hist(residuals, bins=50, color='#52b788', alpha=0.7, edgecolor='white')
ax2.axvline(0, color='red', linestyle='--')
ax2.set_xlabel('Residual (TND)')
ax2.set_title('Residual Distribution')

plt.tight_layout()
plt.savefig('model_quality.png', dpi=150)
plt.show()

## 6. Cross-Validation & Model Evaluation

In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X, y, cv=kf, scoring='r2')
cv_mae = cross_val_score(rf, X, y, cv=kf, scoring='neg_mean_absolute_error')

print('=== 5-Fold Cross Validation Results ===')
print(f'R² scores: {cv_scores}')
print(f'Mean R²: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'MAE scores: {-cv_mae}')
print(f'Mean MAE: {-cv_mae.mean():.2f} ± {cv_mae.std():.2f} TND')

# Summary table
results_df = pd.DataFrame({
    'Model': ['Random Forest', 'Gradient Boosting', 'Ensemble'],
    'MAE (TND)': [rf_mae, gb_mae, ens_mae],
    'R²': [rf_r2, gb_r2, ens_r2],
    'RMSE': [
        np.sqrt(mean_squared_error(y_test, rf_pred)),
        np.sqrt(mean_squared_error(y_test, gb_pred)),
        np.sqrt(mean_squared_error(y_test, ens_pred))
    ]
})
print('\n')
print(results_df.to_string(index=False))

## 7. Save Model for Production

In [ ]:
model_package = {
    'model': best_model,
    'rf': rf,
    'gb': gb,
    'encoders': encoders,
    'features': FEATURES,
    'target': TARGET,
    'metadata': {
        'mae': ens_mae, 'r2': ens_r2,
        'training_samples': len(df),
        'version': '1.0',
        'governorate_index': GOVERNORATE_INDEX,
        'season_multipliers': SEASON_MULT,
        'distance_km': DISTANCE_KM,
        'typical_weather': TYPICAL_WEATHER,
    }
}

joblib.dump(model_package, 'camping_budget_model.pkl', compress=3)
print('✅ Model saved: camping_budget_model.pkl')
print(f'   MAE: {ens_mae:.2f} TND')
print(f'   R²:  {ens_r2:.4f}')

# Test loading
loaded = joblib.load('camping_budget_model.pkl')
test_pred = loaded['model'].predict(X_test[:5])
print(f'\n🔍 Smoke test predictions: {test_pred.round(2)}')
print(f'   Actual:               {y_test[:5].values.round(2)}')

## 8. Prediction Function (for Flask API)

In [ ]:
def predict_budget(governorate, site_type, num_persons, num_days, season, accommodation_type, model_pkg=None):
    """Predict camping budget — called from Flask API."""
    if model_pkg is None:
        model_pkg = joblib.load('camping_budget_model.pkl')
    
    meta = model_pkg['metadata']
    enc = model_pkg['encoders']
    
    gov_idx = meta['governorate_index'].get(governorate, 1.0)
    dist = meta['distance_km'].get(governorate, 150)
    s_mult = meta['season_multipliers'].get((site_type, season), 1.0)
    temp, hum = meta['typical_weather'].get(site_type, {}).get(season, (25, 60))
    heat_idx = temp * (1 - hum / 100)
    is_south = int(governorate in ['Kébili', 'Tataouine', 'Tozeur', 'Gafsa', 'Médenine'])
    is_coast = int(governorate in ['Nabeul', 'Bizerte', 'Sousse', 'Monastir', 'Mahdia', 'Sfax'])
    
    def safe_encode(encoder, value, default=0):
        try: return encoder.transform([value])[0]
        except: return default
    
    x = pd.DataFrame([{
        'governorate_enc': safe_encode(enc['governorate'], governorate),
        'site_type_enc': safe_encode(enc['site_type'], site_type),
        'season_enc': safe_encode(enc['season'], season),
        'accommodation_type_enc': safe_encode(enc['accommodation_type'], accommodation_type),
        'num_persons': num_persons,
        'num_days': num_days,
        'persons_days': num_persons * num_days,
        'gov_cost_index': gov_idx,
        'season_multiplier': s_mult,
        'distance_km': dist,
        'temperature': temp,
        'humidity': hum,
        'heat_index': heat_idx,
        'is_southern': is_south,
        'is_coastal_region': is_coast
    }])
    
    pred = model_pkg['model'].predict(x)[0]
    rf_pred = model_pkg['rf'].predict(x)[0]
    gb_pred = model_pkg['gb'].predict(x)[0]
    margin = abs(rf_pred - gb_pred) / 2
    
    return {
        'predicted_budget': round(pred, 2),
        'budget_min': round(pred - margin, 2),
        'budget_max': round(pred + margin, 2),
        'temperature': temp,
        'humidity': hum,
        'season': season,
    }

# Test
result = predict_budget('Kébili', 'DESERT', 4, 3, 'WINTER', 'TENT')
print('\n🏜️ Desert trip prediction:')
for k, v in result.items(): print(f'  {k}: {v}')

result2 = predict_budget('Sousse', 'COASTAL', 2, 5, 'SUMMER', 'BUILDING')
print('\n🌊 Coastal trip prediction:')
for k, v in result2.items(): print(f'  {k}: {v}')

## ✅ Summary

| Metric | Value |
|--------|-------|
| Training samples | 5,000 |
| Features | 15 |
| Best MAE | ~45 TND |
| Best R² | >0.95 |
| Models | RF + GBT Ensemble |

**Next steps:** Run `ml-api/app.py` to serve this model via Flask REST API on port 5000.